# DNN

PyTorch, CPU-only. Пункты идут по чеклисту: простая MLP, число слоёв, BatchNorm, Dropout, размеры слоёв и активации, оптимизаторы, scheduler, гиперпараметры обучения, опционально Embedding для категориальных фичей

Сети на 1458 домах обучаются быстро, поэтому каждый вариант проверяем через кросс-валидацию KFold с 5 фолдами, а не через один holdout-сплит. Это 5 обучений на вариант вместо 15 у RepeatedKFold с тремя повторами из прошлых ноутбуков, поэтому числа сопоставимы с остальными моделями по порядку величины, но не один к одному. Препроцессинг подгоняется заново внутри каждого фолда

Предобработка признаков, `build_preprocessor` и `rmse` подключаются из модулей `preprocessing.py` и `validation.py` в корне репозитория

## 1. Признаки из прошлых ноутбуков

Те же шаги, что и в `classical_models.ipynb` и `optuna_tuning.ipynb`

In [1]:
import sys
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder, StandardScaler

sys.path.append("..")

from preprocessing import OUTLIER_IDS, apply_accepted_preprocessing, build_preprocessor
from validation import rmse

pd.set_option("display.max_columns", 100)

DATA_DIR = Path("../data")
TARGET = "SalePrice"
RANDOM_STATE = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
X = train.drop(columns=[TARGET, "Id"])
y = train[TARGET]

In [3]:
X_base = apply_accepted_preprocessing(X, DATA_DIR / "train.csv")

keep = ~train["Id"].isin(OUTLIER_IDS)
X_base, y_base = X_base[keep].reset_index(drop=True), y[keep].reset_index(drop=True)
log_y_base = np.log1p(y_base)

print(f"признаков: {X_base.shape[1]}, домов: {X_base.shape[0]}")

признаков: 69, домов: 1458


## 2. Схема валидации и цикл обучения

Категории кодируем one-hot, числа масштабируем, как в `build_preprocessor(scale=True)` из прошлых ноутбуков, но с плотным выходом one-hot, тензорам нужен плотный массив

Таргет, логарифм цены, лежит около 12, а выход сети в начале обучения около нуля. Поэтому внутри каждого фолда стандартизуем таргет по обучающей части, обучаем сеть на стандартизованном таргете и возвращаем предсказание в исходный масштаб перед подсчётом RMSE, так что метрика остаётся RMSE на логарифме цены, как в остальных ноутбуках

Функция `cross_validate_nn` принимает функции, создающие модель, оптимизатор и scheduler, и для каждого из 5 фолдов заново подгоняет препроцессинг, создаёт и обучает модель. Так один и тот же цикл обучения переиспользуется во всех следующих экспериментах, а сети в каждом фолде стартуют с одинаковых весов

In [4]:
build_dnn_preprocessor = partial(build_preprocessor, scale=True, dense=True)


results = []

In [5]:
def train_nn(model, train_loader, optimizer, n_epochs, scheduler=None, loss_fn=None):
    """Обучает модель заданное число эпох"""
    loss_fn = loss_fn or nn.MSELoss()
    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        if scheduler is not None:
            scheduler.step()


def cross_validate_nn(
    experiment,
    build_model,
    build_optimizer,
    n_epochs=100,
    batch_size=32,
    build_scheduler=None,
    loss_fn=None,
    build_features=build_dnn_preprocessor,
):
    """Считает RMSE на логарифме цены по KFold и записывает результат в results

    build_model получает число входных признаков, build_optimizer получает параметры модели,
    build_scheduler получает оптимизатор, build_features создаёт препроцессор признаков. Сеть обучается на стандартизованном таргете, RMSE
    считается после возврата предсказаний в масштаб логарифма цены
    """
    folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for fit_idx, valid_idx in folds.split(X_base):
        preprocessor = build_features()
        X_fit = torch.tensor(preprocessor.fit_transform(X_base.iloc[fit_idx]).astype("float32"))
        X_valid = torch.tensor(preprocessor.transform(X_base.iloc[valid_idx]).astype("float32"))
        y_fit_raw = log_y_base.iloc[fit_idx].values
        y_mean, y_std = y_fit_raw.mean(), y_fit_raw.std()
        y_fit = torch.tensor((y_fit_raw - y_mean) / y_std, dtype=torch.float32).unsqueeze(1)

        torch.manual_seed(RANDOM_STATE)
        train_loader = DataLoader(TensorDataset(X_fit, y_fit), batch_size=batch_size, shuffle=True)
        model = build_model(X_fit.shape[1]).to(device)
        optimizer = build_optimizer(model.parameters())
        scheduler = build_scheduler(optimizer) if build_scheduler else None
        train_nn(model, train_loader, optimizer, n_epochs, scheduler, loss_fn)

        model.eval()
        with torch.no_grad():
            prediction = model(X_valid.to(device)).cpu().numpy().ravel() * y_std + y_mean
        scores.append(rmse(log_y_base.iloc[valid_idx].values, prediction))

    results.append({"experiment": experiment, "rmse": float(np.mean(scores)), "std": float(np.std(scores))})
    print(f"{experiment}: RMSE {np.mean(scores):.4f}, std {np.std(scores):.4f}")


def show_results() -> pd.DataFrame:
    """Собирает результаты всех экспериментов в таблицу, в порядке добавления"""
    results_df = pd.DataFrame(results)
    experiment_order = results_df["experiment"].drop_duplicates()
    return results_df.set_index("experiment").loc[experiment_order, ["rmse", "std"]].round(4)

## 3. Простая MLP

Пункт 1 чеклиста: два линейных слоя с функцией активации между ними. На выходе ничего не применяем, обучаем сразу на MSE от логарифма цены

In [6]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=16):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.activation(x)
        return self.fc2(x)


adam = partial(torch.optim.Adam, lr=1e-3)

cross_validate_nn("1. Простая MLP 16", SimpleMLP, adam)

1. Простая MLP 16: RMSE 0.1344, std 0.0130


### Выводы: простая MLP

- RMSE 0.1344, std 0.0130 по 5 фолдам, один скрытый слой из 16 нейронов без какой-либо настройки
- Это лучше KNN с 0.1657 и одного дерева с 0.1853 из прошлых ноутбуков, на уровне случайного леса с 0.1318 и 0.1376, но хуже бустингов, 0.1150 у LightGBM и 0.1159 у XGBoost через Optuna, хуже обычной линейной регрессии с 0.1198, а Ridge и Lasso с 0.1117 и 0.1111 впереди с заметным отрывом
- Сравнение с прошлыми ноутбуками приблизительное, там использовался RepeatedKFold с тремя повторами, здесь один KFold, а std по фолдам 0.0130 сопоставим с разницей между соседними моделями

## 4. Число слоёв

Пункт 2 чеклиста: добавим больше слоёв. Ширину фиксируем на 16 (как в SimpleMLP из пункта 1) и меняем только глубину, чтобы не путать эффект от числа слоёв с эффектом от их размера, размер отдельно проверим в пункте 5. Обобщаем SimpleMLP до произвольного числа скрытых слоёв через hidden_dims

In [7]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(16,), activation=nn.ReLU):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(activation())
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


architectures = {
    "2. MLP 16-16": (16, 16),
    "2. MLP 16-16-16": (16, 16, 16),
    "2. MLP 16-16-16-16": (16, 16, 16, 16),
}

for name, hidden_dims in architectures.items():
    cross_validate_nn(name, partial(MLP, hidden_dims=hidden_dims), adam)

2. MLP 16-16: RMSE 0.1379, std 0.0062


2. MLP 16-16-16: RMSE 0.1410, std 0.0043


2. MLP 16-16-16-16: RMSE 0.1425, std 0.0151


Продолжаем наращивать глубину до 8 слоёв при той же ширине 16

In [8]:
deeper_architectures = {
    f"2. MLP {n_layers} слоёв по 16": (16,) * n_layers for n_layers in (5, 6, 7, 8)
}

for name, hidden_dims in deeper_architectures.items():
    cross_validate_nn(name, partial(MLP, hidden_dims=hidden_dims), adam)

2. MLP 5 слоёв по 16: RMSE 0.1425, std 0.0039


2. MLP 6 слоёв по 16: RMSE 0.1386, std 0.0101


2. MLP 7 слоёв по 16: RMSE 0.1358, std 0.0106


2. MLP 8 слоёв по 16: RMSE 0.1287, std 0.0105


### Выводы: число слоёв

- От 2 до 5 слоёв RMSE растёт: 0.1379, 0.1410, 0.1425 и 0.1425, все хуже однослойной сети с 0.1344. Дальше идёт спад: 6 слоёв дают 0.1386, 7 слоёв 0.1358, 8 слоёв 0.1287
- Восьмислойная сеть лучшая из всех, но выигрыш у однослойной 0.0057 меньше std по фолдам, 0.0105 и 0.0130, поэтому уверенно сказать, что глубина помогает, нельзя. Весь разброс от 0.1287 до 0.1425 лежит в основном внутри шума, и явной закономерности между глубиной и качеством нет
- Ширина слоёв зафиксирована на 16, размер слоёв проверим отдельно в пункте 5, поэтому вывод относится только к глубине при такой ширине
- Вывод относится к сетям без регуляризации, при BatchNorm и Dropout глубокие сети могут обучаться лучше, поэтому эти два пункта проверяем не только на одном скрытом слое, но и на нескольких значениях глубины

## 5. BatchNorm

Пункт 3 чеклиста: после каждого линейного слоя добавляем нормализацию по батчу, затем активацию. Проверяем несколько значений глубины при ширине 16, потому что нормализация могла бы помочь именно глубоким сетям, которые без неё обучались хуже. Для сравнения у нас уже есть те же глубины без BatchNorm

In [9]:
class MLPWithBatchNorm(nn.Module):
    def __init__(self, input_dim, hidden_dims=(16,), activation=nn.ReLU):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(activation())
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


for n_layers in (1, 2, 4, 8):
    cross_validate_nn(
        f"3. BatchNorm {n_layers} слоёв по 16",
        partial(MLPWithBatchNorm, hidden_dims=(16,) * n_layers),
        adam,
    )

3. BatchNorm 1 слоёв по 16: RMSE 0.1446, std 0.0018


3. BatchNorm 2 слоёв по 16: RMSE 0.1367, std 0.0082


3. BatchNorm 4 слоёв по 16: RMSE 0.1353, std 0.0056


3. BatchNorm 8 слоёв по 16: RMSE 0.1387, std 0.0080


### Выводы: BatchNorm

- С BatchNorm получилось 0.1446 на одном слое, 0.1367 на двух, 0.1353 на четырёх и 0.1387 на восьми. Без него на тех же глубинах 0.1344, 0.1379, 0.1425 и 0.1287
- BatchNorm немного помогает на двух и четырёх слоях, на 0.0012 и 0.0072, и ухудшает результат на одном и восьми слоях, на 0.0102 и 0.0100. Эти разницы сопоставимы с std по фолдам, которое лежит между 0.0018 и 0.0130, поэтому стабильного эффекта нет
- Лучший результат всего ноутбука по-прежнему у сети без BatchNorm, 8 слоёв с 0.1287, а лучший с BatchNorm 4 слоя с 0.1353
- Dropout проверяем на тех же глубинах, 1, 2, 4 и 8 слоёв

## 6. Dropout

Пункт 4 чеклиста: после активации каждого скрытого слоя добавляем Dropout. Проверяем вероятности 0.1, 0.3 и 0.5 на тех же глубинах, что и BatchNorm, 1, 2, 4 и 8 слоёв по 16 нейронов, потому что регуляризация могла бы помочь именно глубоким сетям

In [10]:
class MLPWithDropout(nn.Module):
    def __init__(self, input_dim, hidden_dims=(16,), activation=nn.ReLU, dropout=0.1):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(activation())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


for n_layers in (1, 2, 4, 8):
    for dropout in (0.1, 0.3, 0.5):
        cross_validate_nn(
            f"4. Dropout {dropout}, {n_layers} слоёв по 16",
            partial(MLPWithDropout, hidden_dims=(16,) * n_layers, dropout=dropout),
            adam,
        )

4. Dropout 0.1, 1 слоёв по 16: RMSE 0.1268, std 0.0089


4. Dropout 0.3, 1 слоёв по 16: RMSE 0.1236, std 0.0090


4. Dropout 0.5, 1 слоёв по 16: RMSE 0.1249, std 0.0088


4. Dropout 0.1, 2 слоёв по 16: RMSE 0.1296, std 0.0093


4. Dropout 0.3, 2 слоёв по 16: RMSE 0.1291, std 0.0025


...

4. Dropout 0.3, 4 слоёв по 16: RMSE 0.1520, std 0.0122


4. Dropout 0.5, 4 слоёв по 16: RMSE 0.1798, std 0.0268


4. Dropout 0.1, 8 слоёв по 16: RMSE 0.1565, std 0.0121


4. Dropout 0.3, 8 слоёв по 16: RMSE 0.1971, std 0.0302


4. Dropout 0.5, 8 слоёв по 16: RMSE 0.2613, std 0.0281


### Выводы: Dropout

Без Dropout на глубинах 1, 2, 4 и 8 слоёв было 0.1344, 0.1379, 0.1425 и 0.1287

| Слоёв | p=0.1 | p=0.3 | p=0.5 |
| --- | --- | --- | --- |
| 1 | 0.1268 | 0.1236 | 0.1249 |
| 2 | 0.1296 | 0.1291 | 0.1401 |
| 4 | 0.1381 | 0.1520 | 0.1798 |
| 8 | 0.1565 | 0.1971 | 0.2613 |

- Dropout помогает мелким сетям: на одном слое все три вероятности лучше, чем без него, лучшая p=0.3 с 0.1236, а на двух слоях помогают p=0.1 и p=0.3, на 0.0083 и 0.0088, при p=0.5 результат хуже
- Глубоким сетям Dropout вредит, и тем сильнее, чем больше вероятность: на четырёх слоях лучше без него только при p=0.1, на восьми хуже все три варианта, и RMSE растёт от 0.1565 до 0.2613 вместе с p. Предположение, что регуляризация вернёт пользу глубоким сетям, не подтвердилось
- Возможная причина в ширине слоёв, при 16 нейронах и p=0.5 в каждом слое остаётся в среднем 8 активных нейронов, и через 8 таких слоёв сигнал портится сильнее, чем помогает регуляризация. Это объяснение по арифметике ширины, отдельно мы его не проверяли
- 0.1236 на одном слое с Dropout 0.3 теперь лучший результат ноутбука, прежний лучший был 0.1287 у восьми слоёв без регуляризации. Разница с соседними вариантами на одном слое, 0.1268 и 0.1249, меньше std по фолдам, около 0.009, поэтому выбор именно p=0.3 неустойчив, надёжен только вывод, что на одном слое Dropout лучше, чем без него
- Это всё ещё хуже обычной линейной регрессии с 0.1198 и бустингов, а до Ridge и Lasso с 0.1117 и 0.1111 остаётся заметный отрыв

## 7. Размер слоёв

Пункт 5 чеклиста, первая часть: ширина скрытых слоёв. Берём Dropout 0.3, лучший вариант из прошлого раздела, и проверяем ширины 8, 32, 64, 128 и 256 на одном и двух скрытых слоях. Ширина 16 уже есть в прошлом разделе, 0.1236 на одном слое и 0.1291 на двух

In [11]:
width_options = (8, 32, 64, 128, 256)

for n_layers in (1, 2):
    for width in width_options:
        cross_validate_nn(
            f"5. Dropout 0.3, {n_layers} слоёв по {width}",
            partial(MLPWithDropout, hidden_dims=(width,) * n_layers, dropout=0.3),
            adam,
        )

5. Dropout 0.3, 1 слоёв по 8: RMSE 0.1256, std 0.0037


5. Dropout 0.3, 1 слоёв по 32: RMSE 0.1244, std 0.0099


5. Dropout 0.3, 1 слоёв по 64: RMSE 0.1238, std 0.0087


5. Dropout 0.3, 1 слоёв по 128: RMSE 0.1243, std 0.0113


5. Dropout 0.3, 1 слоёв по 256: RMSE 0.1239, std 0.0102


5. Dropout 0.3, 2 слоёв по 8: RMSE 0.1574, std 0.0116


5. Dropout 0.3, 2 слоёв по 32: RMSE 0.1291, std 0.0064


5. Dropout 0.3, 2 слоёв по 64: RMSE 0.1239, std 0.0106


5. Dropout 0.3, 2 слоёв по 128: RMSE 0.1226, std 0.0097


5. Dropout 0.3, 2 слоёв по 256: RMSE 0.1225, std 0.0102


### Выводы: размер слоёв

Ширина 16 взята из прошлого раздела с тем же Dropout 0.3

| Ширина | 1 слой | 2 слоя |
| --- | --- | --- |
| 8 | 0.1256 | 0.1574 |
| 16 | 0.1236 | 0.1291 |
| 32 | 0.1244 | 0.1291 |
| 64 | 0.1238 | 0.1239 |
| 128 | 0.1243 | 0.1226 |
| 256 | 0.1239 | 0.1225 |

- На одном слое ширина почти не влияет: от 0.1236 до 0.1256, а std по фолдам около 0.009 и выше, разница между ширинами лежит внутри шума. Слой из 8 нейронов не хуже слоя из 256
- На двух слоях ширина важна: RMSE падает от 0.1574 при ширине 8 до 0.1239 при 64 и дальше почти не меняется, 0.1226 при 128 и 0.1225 при 256. Слишком узкая сеть с Dropout 0.3 работает заметно хуже, что согласуется с предположением из прошлого раздела про потерю сигнала при малой ширине, хотя прямо мы его не проверяли
- 0.1225 у двух слоёв по 256 формально лучший результат ноутбука, но отличается от 0.1236 у одного слоя с шириной 16 всего на 0.0011, это ничтожно мало по сравнению с std по фолдам, поэтому считаем эти две конфигурации равными по качеству, а более простую, один слой, надёжнее
- Даже лучшие варианты остаются хуже обычной линейной регрессии с 0.1198 и бустингов, а до Ridge и Lasso с 0.1117 и 0.1111 остаётся заметный отрыв

## 8. Функции активации

Пункт 5 чеклиста, вторая часть: функции активации. Берём один скрытый слой шириной 64 с Dropout 0.3, на такой ширине результат из прошлого раздела уже вышел на плато. ReLU для этой конфигурации уже посчитан, 0.1238, проверяем Tanh, GELU, LeakyReLU, ELU, SiLU и Sigmoid

In [12]:
activation_options = {
    "Tanh": nn.Tanh,
    "GELU": nn.GELU,
    "LeakyReLU": nn.LeakyReLU,
    "ELU": nn.ELU,
    "SiLU": nn.SiLU,
    "Sigmoid": nn.Sigmoid,
}

for name, activation in activation_options.items():
    cross_validate_nn(
        f"5. Активация {name}, 1 слой по 64",
        partial(MLPWithDropout, hidden_dims=(64,), activation=activation, dropout=0.3),
        adam,
    )

5. Активация Tanh, 1 слой по 64: RMSE 0.1197, std 0.0053


5. Активация GELU, 1 слой по 64: RMSE 0.1288, std 0.0071


5. Активация LeakyReLU, 1 слой по 64: RMSE 0.1237, std 0.0089


5. Активация ELU, 1 слой по 64: RMSE 0.1176, std 0.0065


5. Активация SiLU, 1 слой по 64: RMSE 0.1242, std 0.0081


5. Активация Sigmoid, 1 слой по 64: RMSE 0.1199, std 0.0056


### Выводы: функции активации

Один скрытый слой шириной 64 с Dropout 0.3, ReLU из прошлого раздела дал 0.1238

| Активация | RMSE | std |
| --- | --- | --- |
| ELU | 0.1176 | 0.0065 |
| Tanh | 0.1197 | 0.0053 |
| Sigmoid | 0.1199 | 0.0056 |
| LeakyReLU | 0.1237 | 0.0089 |
| ReLU | 0.1238 | 0.0087 |
| SiLU | 0.1242 | 0.0081 |
| GELU | 0.1288 | 0.0071 |

- Активации делятся на две группы. ELU, Tanh и Sigmoid дают 0.1176, 0.1197 и 0.1199, а ReLU и близкие к ней LeakyReLU, SiLU и GELU дают от 0.1237 до 0.1288. Разница между группами около 0.004 до 0.011, что сопоставимо со std по фолдам, но группы разделяются чисто, ни одна ReLU-подобная активация не вошла в число трёх лучших
- ELU лучшая, 0.1176, что на 0.0062 лучше ReLU, и это новый лучший результат ноутбука, прежний был 0.1225. Между ELU, Tanh и Sigmoid разница 0.002 и меньше, отличить их по этому запуску нельзя
- Сравнение с прошлыми ноутбуками приблизительное из-за разной схемы валидации, но по порядку величины ELU с 0.1176 уже лучше обычной линейной регрессии с 0.1198 и близка к CatBoost с 0.1165 и бустингам из Optuna с 0.1150 и 0.1159, при этом до Ridge и Lasso с 0.1117 и 0.1111 остаётся заметный отрыв
- Дальше берём ELU, один слой шириной 64 и Dropout 0.3 как текущую основу для оптимизаторов и scheduler

## 9. Оптимизаторы

Пункт 6 чеклиста: разные оптимизаторы. Основа та же, что после проверки активаций, один скрытый слой шириной 64 с ELU и Dropout 0.3. Adam с lr 1e-3 для неё уже посчитан, 0.1176. Каждому оптимизатору даём типичный для него шаг обучения и отдельно его не подбираем, шаг обучения проверим в следующем разделе

In [13]:
optimizer_options = {
    "SGD": partial(torch.optim.SGD, lr=1e-2),
    "SGD с моментом 0.9": partial(torch.optim.SGD, lr=1e-2, momentum=0.9),
    "RMSprop": partial(torch.optim.RMSprop, lr=1e-3),
    "AdamW": partial(torch.optim.AdamW, lr=1e-3),
    "NAdam": partial(torch.optim.NAdam, lr=1e-3),
    "Adagrad": partial(torch.optim.Adagrad, lr=1e-2),
}

for name, build_optimizer in optimizer_options.items():
    cross_validate_nn(
        f"6. Оптимизатор {name}",
        partial(MLPWithDropout, hidden_dims=(64,), activation=nn.ELU, dropout=0.3),
        build_optimizer,
    )

6. Оптимизатор SGD: RMSE 0.1150, std 0.0053


6. Оптимизатор SGD с моментом 0.9: RMSE 0.1167, std 0.0061


6. Оптимизатор RMSprop: RMSE 0.1192, std 0.0085


6. Оптимизатор AdamW: RMSE 0.1176, std 0.0065


6. Оптимизатор NAdam: RMSE 0.1176, std 0.0069


6. Оптимизатор Adagrad: RMSE 0.1152, std 0.0053


### Выводы: оптимизаторы

Один скрытый слой шириной 64 с ELU и Dropout 0.3, Adam с lr 1e-3 дал 0.1176

| Оптимизатор | lr | RMSE | std |
| --- | --- | --- | --- |
| SGD | 1e-2 | 0.1150 | 0.0053 |
| Adagrad | 1e-2 | 0.1152 | 0.0053 |
| SGD с моментом 0.9 | 1e-2 | 0.1167 | 0.0061 |
| Adam | 1e-3 | 0.1176 | 0.0065 |
| AdamW | 1e-3 | 0.1176 | 0.0065 |
| NAdam | 1e-3 | 0.1176 | 0.0069 |
| RMSprop | 1e-3 | 0.1192 | 0.0085 |

- Лучшие результаты у обычного SGD, 0.1150, и Adagrad, 0.1152, они неразличимы между собой. Adam, AdamW и NAdam дают одинаковые 0.1176, а RMSprop хуже всех, 0.1192
- Разница между лучшим и худшим оптимизатором 0.0042, что меньше std по фолдам, поэтому выигрыш SGD и Adagrad нельзя считать надёжным, хотя оба показывают самую низкую std, 0.0053
- Сравнение оптимизаторов здесь смешано с разницей в шаге обучения: SGD и Adagrad получили lr 1e-2, остальные 1e-3. Отделить эффект оптимизатора от эффекта шага обучения по этому эксперименту нельзя, шаг обучения проверим отдельно
- Лучший результат ноутбука теперь 0.1150 у SGD, по порядку величины это на уровне LightGBM через Optuna с 0.1150 и лучше XGBoost и CatBoost с 0.1159 и 0.1165, при приблизительном сравнении из-за разной схемы валидации. До Ridge и Lasso с 0.1117 и 0.1111 отрыв сохраняется
- Для scheduler берём SGD, так как он проще Adagrad при том же качестве

## 10. Scheduler

Пункт 7 чеклиста: косинусный scheduler. Основа: один скрытый слой шириной 64 с ELU и Dropout 0.3 и SGD с lr 1e-2, без scheduler для неё получено 0.1150. Проверяем три варианта: косинусное затухание lr до нуля за все 100 эпох, то же затухание до 1e-4 и косинус с перезапусками каждые 25 эпох

In [14]:
from torch.optim.lr_scheduler import CosineAnnealingLR, CosineAnnealingWarmRestarts

scheduler_options = {
    "косинус до 0": partial(CosineAnnealingLR, T_max=100),
    "косинус до 1e-4": partial(CosineAnnealingLR, T_max=100, eta_min=1e-4),
    "косинус с перезапусками каждые 25 эпох": partial(CosineAnnealingWarmRestarts, T_0=25),
}

for name, build_scheduler in scheduler_options.items():
    cross_validate_nn(
        f"7. Scheduler {name}",
        partial(MLPWithDropout, hidden_dims=(64,), activation=nn.ELU, dropout=0.3),
        partial(torch.optim.SGD, lr=1e-2),
        build_scheduler=build_scheduler,
    )

7. Scheduler косинус до 0: RMSE 0.1155, std 0.0060


7. Scheduler косинус до 1e-4: RMSE 0.1155, std 0.0059


7. Scheduler косинус с перезапусками каждые 25 эпох: RMSE 0.1153, std 0.0059


### Выводы: scheduler

Один скрытый слой шириной 64 с ELU и Dropout 0.3, SGD с lr 1e-2, без scheduler получено 0.1150 со std 0.0053

| Scheduler | RMSE | std |
| --- | --- | --- |
| Без scheduler | 0.1150 | 0.0053 |
| Косинус с перезапусками каждые 25 эпох | 0.1153 | 0.0059 |
| Косинус до 0 | 0.1155 | 0.0060 |
| Косинус до 1e-4 | 0.1155 | 0.0059 |

- Ни один вариант не улучшил результат, все три на 0.0003 и 0.0005 хуже, чем без scheduler, что примерно на порядок меньше std по фолдам, поэтому фактически разницы нет
- Три варианта косинуса не отличаются друг от друга: затухание до нуля, до 1e-4 и с перезапусками дают 0.1153 и 0.1155. Почему scheduler здесь не даёт эффекта, мы не выясняли
- Scheduler в дальнейших экспериментах не используем, основа остаётся без него: один скрытый слой шириной 64 с ELU и Dropout 0.3 и SGD с lr 1e-2

## 11. Параметры обучения

Пункт 8 чеклиста: шаг обучения, размер батча, число эпох и функция потерь. Основа: один скрытый слой шириной 64 с ELU и Dropout 0.3, SGD с lr 1e-2, батч 32, 100 эпох, MSE, без scheduler, для неё получено 0.1150. Каждый параметр меняем по отдельности, остальные держим на значениях основы

In [15]:
base_model = partial(MLPWithDropout, hidden_dims=(64,), activation=nn.ELU, dropout=0.3)

### Шаг обучения

Основа использует lr 1e-2, проверяем 3e-3, 3e-2 и 1e-1

In [16]:
lr_options = (3e-3, 3e-2, 1e-1)

for lr in lr_options:
    cross_validate_nn(f"8. lr {lr}", base_model, partial(torch.optim.SGD, lr=lr))

8. lr 0.003: RMSE 0.1171, std 0.0059


8. lr 0.03: RMSE 0.1155, std 0.0062


8. lr 0.1: RMSE 0.1259, std 0.0133


### Размер батча

Основа использует батч 32, проверяем 16, 64 и 128

In [17]:
batch_size_options = (16, 64, 128)

for batch_size in batch_size_options:
    cross_validate_nn(
        f"8. батч {batch_size}", base_model, partial(torch.optim.SGD, lr=1e-2), batch_size=batch_size
    )

8. батч 16: RMSE 0.1148, std 0.0053


8. батч 64: RMSE 0.1158, std 0.0055


8. батч 128: RMSE 0.1177, std 0.0059


### Число эпох

Основа обучается 100 эпох, проверяем 50, 200 и 400

In [18]:
epoch_options = (50, 200, 400)

for n_epochs in epoch_options:
    cross_validate_nn(
        f"8. эпох {n_epochs}", base_model, partial(torch.optim.SGD, lr=1e-2), n_epochs=n_epochs
    )

8. эпох 50: RMSE 0.1152, std 0.0061


8. эпох 200: RMSE 0.1138, std 0.0057


8. эпох 400: RMSE 0.1136, std 0.0056


### Функция потерь

Основа обучается на MSE, метрика ноутбука тоже RMSE. Проверяем L1 и Huber с порогом 1.0, таргет стандартизован, поэтому порог равен одному стандартному отклонению логарифма цены

In [19]:
loss_options = {"L1": nn.L1Loss(), "Huber": nn.HuberLoss(delta=1.0)}

for name, loss_fn in loss_options.items():
    cross_validate_nn(
        f"8. потери {name}", base_model, partial(torch.optim.SGD, lr=1e-2), loss_fn=loss_fn
    )

8. потери L1: RMSE 0.1148, std 0.0076


8. потери Huber: RMSE 0.1156, std 0.0060


### Выводы: параметры обучения

Основа дала 0.1150 со std 0.0053: SGD с lr 1e-2, батч 32, 100 эпох, MSE

| Параметр | Значение | RMSE | std |
| --- | --- | --- | --- |
| lr | 3e-3 | 0.1171 | 0.0059 |
| lr | 1e-2, основа | 0.1150 | 0.0053 |
| lr | 3e-2 | 0.1155 | 0.0062 |
| lr | 1e-1 | 0.1259 | 0.0133 |
| батч | 16 | 0.1148 | 0.0053 |
| батч | 32, основа | 0.1150 | 0.0053 |
| батч | 64 | 0.1158 | 0.0055 |
| батч | 128 | 0.1177 | 0.0059 |
| эпох | 50 | 0.1152 | 0.0061 |
| эпох | 100, основа | 0.1150 | 0.0053 |
| эпох | 200 | 0.1138 | 0.0057 |
| эпох | 400 | 0.1136 | 0.0056 |
| потери | MSE, основа | 0.1150 | 0.0053 |
| потери | L1 | 0.1148 | 0.0076 |
| потери | Huber | 0.1156 | 0.0060 |

- Шаг обучения: 1e-2 и 3e-2 дают одинаковый результат, 0.1150 и 0.1155, при 3e-3 хуже, 0.1171, а при 1e-1 обучение становится нестабильным, RMSE 0.1259 и std растёт до 0.0133, то есть в основе шаг взят удачно, в середине рабочего диапазона
- Размер батча: RMSE растёт вместе с батчем, от 0.1148 при 16 до 0.1177 при 128. Разница между крайними значениями 0.0029 меньше std по фолдам, но тенденция одна и та же на всех четырёх значениях. Меньший батч при том же числе эпох означает больше шагов оптимизатора
- Число эпох: 200 и 400 эпох дают 0.1138 и 0.1136 против 0.1150 на 100, улучшение есть, но оно на 0.0012 и 0.0014, то есть заметно меньше std, и после 200 эпох результат почти не меняется
- Функция потерь на результат не влияет: L1 даёт 0.1148, Huber 0.1156, MSE 0.1150, все внутри шума, при этом L1 самая нестабильная по фолдам, std 0.0076
- Лучший результат ноутбука теперь 0.1136 при 400 эпохах, но отличие от основы находится внутри шума, и по этому пункту нельзя утверждать, что какая-то настройка надёжно улучшила сеть. Ridge и Lasso с 0.1117 и 0.1111 остаются впереди, при приблизительном сравнении из-за разной схемы валидации

## 12. Embedding для категориальных фичей

Пункт 9 чеклиста, со звёздочкой: вместо one-hot каждую категориальную колонку кодируем числовым индексом и пропускаем через свой слой Embedding, вектора всех категорий склеиваем с масштабированными числовыми признаками и передаём в тот же блок из одного скрытого слоя, что и в основе. Индекс 0 отдан категориям, которых не было в обучающей части фолда

Препроцессор отдаёт одну матрицу, сначала числовые колонки, потом индексы категорий, а сеть сама разрезает её по числу числовых колонок. Так функция `cross_validate_nn` остаётся той же, только с другим препроцессором. Всё остальное как в основе: один скрытый слой шириной 64 с ELU и Dropout 0.3, SGD с lr 1e-2, батч 32, 100 эпох, для которой с one-hot получено 0.1150. Размер вектора эмбеддинга у всех колонок одинаковый, проверяем 2, 4 и 8

In [20]:
numeric_columns = X_base.select_dtypes("number").columns.tolist()
categorical_columns = X_base.select_dtypes(exclude="number").columns.tolist()
cardinalities = [X_base[column].nunique() + 1 for column in categorical_columns]


def build_embedding_preprocessor() -> ColumnTransformer:
    """Числа масштабируем, категории кодируем индексами от 1, неизвестные категории получают 0"""
    return ColumnTransformer(
        [
            ("numeric", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_columns),
            (
                "categorical",
                make_pipeline(
                    SimpleImputer(strategy="most_frequent"),
                    OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
                    FunctionTransformer(lambda codes: codes + 1),
                ),
                categorical_columns,
            ),
        ]
    )


class EmbeddingMLP(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dims=(64,), activation=nn.ELU, dropout=0.3):
        super().__init__()
        self.n_numeric = len(numeric_columns)
        self.embeddings = nn.ModuleList(nn.Embedding(size, embedding_dim) for size in cardinalities)
        head_input_dim = self.n_numeric + embedding_dim * len(cardinalities)
        self.head = MLPWithDropout(head_input_dim, hidden_dims, activation, dropout)

    def forward(self, x):
        numeric = x[:, : self.n_numeric]
        codes = x[:, self.n_numeric :].long()
        embedded = [embedding(codes[:, i]) for i, embedding in enumerate(self.embeddings)]
        return self.head(torch.cat([numeric, *embedded], dim=1))


print(f"числовых колонок: {len(numeric_columns)}, категориальных: {len(categorical_columns)}")

числовых колонок: 39, категориальных: 30


In [21]:
embedding_dim_options = (2, 4, 8)

for embedding_dim in embedding_dim_options:
    cross_validate_nn(
        f"9. Embedding {embedding_dim}",
        partial(EmbeddingMLP, embedding_dim=embedding_dim),
        partial(torch.optim.SGD, lr=1e-2),
        build_features=build_embedding_preprocessor,
    )

9. Embedding 2: RMSE 0.1186, std 0.0067


9. Embedding 4: RMSE 0.1203, std 0.0073


9. Embedding 8: RMSE 0.1149, std 0.0074


### Выводы: Embedding

Признаков 69, из них 39 числовых и 30 категориальных. Основа с one-hot дала 0.1150 со std 0.0053

| Размер вектора | RMSE | std |
| --- | --- | --- |
| 2 | 0.1186 | 0.0067 |
| 4 | 0.1203 | 0.0073 |
| 8 | 0.1149 | 0.0074 |

- Embedding не улучшил результат: при размере 8 получилось 0.1149, то есть то же, что и с one-hot, а при 2 и 4 хуже на 0.0036 и 0.0053
- Зависимость от размера вектора немонотонная, 4 хуже, чем 2 и 8, и std по фолдам у всех трёх вариантов выше, чем у основы, 0.0067 и 0.0074 против 0.0053, поэтому различия между размерами вектора лежат внутри шума
- Причины, по которым Embedding здесь не даёт выигрыша, мы не проверяли. Проверена только одна конфигурация остальных параметров, взятая из основы, а не подобранная отдельно под Embedding

## Итоги ноутбука

Прошли все пункты чеклиста для DNN: простая MLP, число слоёв, BatchNorm, Dropout, размеры слоёв и активации, оптимизаторы, scheduler, шаг обучения, размер батча, число эпох, функция потерь и Embedding для категориальных фичей. Каждый вариант считался через KFold с 5 фолдами, препроцессинг и стандартизация таргета заново внутри каждого фолда. Следующие шаги строились на лучшем варианте предыдущих

Лучший вариант по каждому пункту:

| Пункт | Лучший вариант | RMSE | std |
| --- | --- | --- | --- |
| 1. Простая MLP | один слой из 16 нейронов | 0.1344 | 0.0130 |
| 2. Число слоёв | 8 слоёв по 16 | 0.1287 | 0.0105 |
| 3. BatchNorm | 4 слоя по 16 | 0.1353 | 0.0056 |
| 4. Dropout | 1 слой по 16, p=0.3 | 0.1236 | 0.0090 |
| 5. Размер слоёв | 2 слоя по 256 с Dropout 0.3 | 0.1225 | 0.0102 |
| 5. Активации | ELU, 1 слой по 64 с Dropout 0.3 | 0.1176 | 0.0065 |
| 6. Оптимизаторы | SGD с lr 1e-2 | 0.1150 | 0.0053 |
| 7. Scheduler | без scheduler, лучший косинус с перезапусками 0.1153 | 0.1150 | 0.0053 |
| 8. Параметры обучения | 400 эпох | 0.1136 | 0.0056 |
| 9. Embedding | размер вектора 8, не лучше one-hot | 0.1149 | 0.0074 |

Что показали эксперименты:

- **Реально улучшили результат три вещи.** Dropout на неглубокой сети, с 0.1344 до 0.1236, замена ReLU на ELU, Tanh или Sigmoid, до 0.1176 и 0.1199, и переход с Adam на SGD или Adagrad, до 0.1150 и 0.1152. Каждый из этих шагов повторяется на нескольких вариантах внутри своей группы, а не держится на одной точке
- **Ничего не дали** число слоёв, BatchNorm, scheduler, функция потерь и Embedding. Глубина без Dropout шла вверх и вниз без закономерности, Dropout вредил глубоким сетям и тем сильнее, чем выше вероятность, а Embedding при размере 8 совпал с one-hot, 0.1149 против 0.1150
- **Итоговое улучшение от 0.1344 до 0.1136 больше, чем разброс по фолдам**, около 0.013 у первой сети, но многие отдельные шаги, например смена оптимизатора на 0.0026 или 400 эпох вместо 100 на 0.0014, меньше std по фолдам, поэтому конкретный выбор таких значений надёжным считать нельзя
- **Оценка немного оптимистична.** Из 63 вариантов мы выбирали лучший по тем же 5 фолдам, на которых считали качество, поэтому итоговое 0.1136 скорее оптимистичная оценка, на новых данных сеть, вероятно, покажет чуть хуже

Сравнение с моделями из прошлых ноутбуков, у каждой модели лучший результат из `classical_models.ipynb` и `optuna_tuning.ipynb`:

| Модель | Лучший RMSE | Ноутбук |
| --- | --- | --- |
| Lasso | 0.1110 | Optuna |
| ElasticNet | 0.1111 | Optuna и RandomizedSearchCV |
| Ridge | 0.1117 | Optuna и GridSearchCV |
| **DNN** | **0.1136** | этот ноутбук |
| LightGBM | 0.1150 | Optuna |
| XGBoost | 0.1159 | Optuna |
| CatBoost | 0.1165 | RandomizedSearchCV |
| Линейная регрессия | 0.1198 | classical_models |
| Случайный лес | 0.1318 | Optuna |
| KNN | 0.1657 | Optuna |
| Дерево решений | 0.1853 | RandomizedSearchCV |

Лучшая нейросеть встаёт между регуляризованными линейными моделями и бустингами, с бустингами она сопоставима, а до Ridge и Lasso остаётся отрыв 0.0019 и 0.0026. Сравнение приблизительное, в прошлых ноутбуках использовался RepeatedKFold с тремя повторами, здесь один KFold, а разница между моделями в верхней части таблицы меньше std по фолдам, около 0.005 и 0.006

Для ансамбля в следующем ноутбуке имеет смысл проверить нейросеть вместе с Lasso и бустингом, то есть модели из разных семейств